# Lab 2: NumPy und pandas

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Datenordner finden: Notebook liegt in labs/ oder loesungen/, die Daten in data/
DATA = next(p for p in [Path("data"), Path("../data"), Path("../../data")] if p.exists())
print("Datenordner:", DATA)

Dieses Lab gehört zu **Teil 2: NumPy und pandas**. Sie bearbeiten es in sechs kurzen Blöcken zwischen den Folien.

**Lernziele**

- Sie legen NumPy-Arrays an, rechnen vektorisiert und werten sie mit Masken und `axis` aus
- Sie lesen eine CSV-Datei als DataFrame ein und wählen Zeilen und Spalten mit `loc` und `iloc` aus
- Sie filtern mit Masken, `query` und `isin` und leiten neue Spalten ab (BMI, Altersgruppe, Geburtsjahr)
- Sie werten Tabellen mit `groupby` und `agg` aus und wissen, wann der Median besser passt als der Mittelwert
- Sie verbinden zwei Tabellen mit `merge` und speichern ein Ergebnis als CSV

**So arbeiten Sie:** Jede Aufgabe hat eine eigene Codezelle mit einem Gerüst. Ersetzen Sie `...` durch Ihren Code. Unter jeder Aufgabe steht ein Kontrollergebnis, mit dem Sie sich selbst prüfen.

Der Datensatz `versicherte.csv` ist synthetisch: erfundene Versicherte, keine echten Personen.

In [ ]:
import time
import tempfile

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
print("NumPy", np.__version__, "| pandas", pd.__version__)

# Leitdatensatz für die pandas-Blöcke (ab Block 3)
df = pd.read_csv(DATA / "versicherte.csv")
print("Versicherte:", df.shape)

## Block 1: Arrays anlegen und rechnen

1. Legen Sie ein Array mit den Zahlen 0 bis 99 an, quadrieren Sie es und bilden Sie die Summe. **Erwartet: 328350**
2. Rechnen Sie eine Million Temperaturwerte von Celsius nach Fahrenheit um (`F = C * 9 / 5 + 32`). Die Schleife steht schon da. Ergänzen Sie die vektorisierte Fassung und vergleichen Sie die Zeiten. **Erwartet: beide Ergebnisse sind gleich (`True`), Mittelwert 54.5 Grad Fahrenheit, die vektorisierte Fassung ist um ein Vielfaches schneller**
3. Legen Sie `np.linspace(0, 10, 100)` an und geben Sie `shape`, `dtype`, `min` und `max` aus. **Erwartet: (100,), float64, 0.0, 10.0**

In [ ]:
# Aufgabe 1: Zahlen 0 bis 99, quadrieren, Summe bilden
# Tipp: np.arange, ** 2, .sum()
zahlen = ...
summe = ...
summe

In [ ]:
# Aufgabe 2: Celsius nach Fahrenheit, Schleife gegen Vektorisierung
rng = np.random.default_rng(42)
celsius = rng.uniform(-10, 35, size=1_000_000)

# Schleife (fertig)
start = time.time()
f_schleife = np.zeros_like(celsius)
for i in range(celsius.size):
    f_schleife[i] = celsius[i] * 9 / 5 + 32
t_schleife = time.time() - start
print("Schleife:     ", round(t_schleife, 4), "s")

# Vektorisiert (Ihr Teil): dieselbe Formel auf das ganze Array anwenden
start = time.time()
f_vektor = ...
t_vektor = time.time() - start
print("Vektorisiert: ", round(t_vektor, 4), "s")

# Kontrolle, sobald f_vektor berechnet ist:
# print(np.allclose(f_schleife, f_vektor), f_vektor.mean().round(2))

In [ ]:
# Aufgabe 3: linspace untersuchen
# Tipp: shape und dtype sind Attribute (ohne Klammern), min und max sind Methoden
x = ...
# print(x.shape, x.dtype, x.min(), x.max())

## Block 2: Masken, Aggregationen, Zufall

Jede Aufgabe legt ihren Generator neu mit `np.random.default_rng(42)` an. So erhalten Sie dieselben Zahlen wie in den Kontrollergebnissen, egal in welcher Reihenfolge Sie die Zellen ausführen.

1. Erzeugen Sie 1000 normalverteilte Körpergrößen mit `rng.normal(170, 10, 1000)`. Wie groß ist der Anteil über 190 cm? **Erwartet: 0.019 (19 von 1000)**
2. Legen Sie eine Matrix mit 5 Zeilen und 3 Spalten aus Zufallszahlen an (`rng.random((5, 3))`). Berechnen Sie den Mittelwert je Spalte und je Zeile. **Erwartet: je Spalte 3 Werte (gerundet 0.67, 0.5, 0.67), je Zeile 5 Werte (der erste gerundet 0.69)**
3. Erweitern Sie die Würfelsimulation von der Folie auf drei Würfel und 10 000 Würfe. Wie oft ist die Summe mindestens 15? **Erwartet: 0.0908 (Theorie: 20/216 = 0.0926)**

In [ ]:
# Aufgabe 1: Anteil der Körpergrößen über 190 cm
# Tipp: Maske bilden, mean() einer Maske ist der Anteil der Treffer
rng = np.random.default_rng(42)
groessen = ...
anteil = ...
anteil

In [ ]:
# Aufgabe 2: Mittelwert je Spalte und je Zeile
# Tipp: axis=0 fasst die Zeilen zusammen (ein Wert je Spalte), axis=1 die Spalten
rng = np.random.default_rng(42)
matrix = ...
je_spalte = ...
je_zeile = ...
# print(je_spalte.round(2), je_zeile.round(2))

In [ ]:
# Aufgabe 3: drei Würfel, 10 000 Würfe, Summe mindestens 15
# Tipp: rng.integers(1, 7, size=(10_000, 3)), dann sum(axis=1)
rng = np.random.default_rng(42)
wuerfe = ...
summen = ...
anteil_15 = ...
anteil_15

## Block 3: Einlesen und Auswählen

1. Lesen Sie `versicherte.csv` selbst noch einmal in `df` ein und geben Sie `shape`, `head()` und `info()` aus. Welche Spalten sind Text, welche Zahlen, wo fehlen Werte? **Erwartet: (5025, 18), fehlende Werte in beruf, blutgruppe, gewicht_kg und bmi**
2. Wählen Sie die Spalten `vorname`, `nachname`, `stadt` der Zeilen 10 bis 20 aus, einmal mit `loc`, einmal mit `iloc`. **Erwartet: beide Male 11 Zeilen und 3 Spalten, `equals` ergibt `True`**
3. Bestimmen Sie Mittelwert, Minimum und Maximum von `arztbesuche_jahr`. **Erwartet: Mittelwert 5.86, Minimum 0, Maximum 57**

In [ ]:
# Aufgabe 1: Einlesen und Überblick
# Tipp: pd.read_csv(DATA / "versicherte.csv")
df = pd.read_csv(DATA / "versicherte.csv")   # diese Zeile dürfen Sie so lassen
# Ihr Teil: shape, head() und info() ausgeben
...

In [ ]:
# Aufgabe 2: Zeilen 10 bis 20, drei Spalten, mit loc und mit iloc
# Tipp: loc schließt das Ende ein, iloc schließt es aus.
# Die Positionen der Spalten finden Sie mit list(df.columns).
mit_loc = ...
mit_iloc = ...
# print(mit_loc.shape, mit_iloc.shape, mit_loc.equals(mit_iloc))

In [ ]:
# Aufgabe 3: Mittelwert, Minimum, Maximum der Arztbesuche
# Tipp: einzeln mit mean(), min(), max() oder auf einmal mit agg([...])
kennzahlen = ...
kennzahlen

## Block 4: Filtern und neue Spalten

1. Wählen Sie alle männlichen Versicherten mit BMI über 30 aus, einmal mit einer Maske, einmal mit `query`. **Erwartet: beide Male 471 Zeilen**
2. Wählen Sie mit `isin` alle Versicherten aus den Stadtstaaten Berlin, Hamburg und Bremen aus. Wie viele davon rauchen? **Erwartet: 1977 Versicherte, davon 502 Raucherinnen und Raucher**
3. Rechnen Sie den BMI aus `groesse_cm` und `gewicht_kg` nach (Gewicht in kg geteilt durch Größe in m zum Quadrat, auf eine Stelle gerundet) und vergleichen Sie mit der vorhandenen Spalte `bmi`. **Erwartet: größte Abweichung 0.0, in 154 Zeilen fehlt der BMI, weil das Gewicht fehlt**
4. Legen Sie mit `pd.cut` die Spalten `altersgruppe` (Kind unter 18, Erwachsen, Senior ab 65) und `bmi_klasse` (Untergewicht unter 18.5, Normalgewicht unter 25, Übergewicht unter 30, Adipositas) an. Zeigen Sie die Normalgewichtigen nach Alter absteigend sortiert. **Erwartet: 3674 Erwachsene, 1351 Senioren, 0 Kinder; 1875 Normalgewichtige, die ältesten sind 99**

In [ ]:
# Aufgabe 1: männlich und BMI über 30, mit Maske und mit query
# Tipp: jede Bedingung in Klammern, verknüpft mit &. In query: and, Text in einfachen Anführungszeichen
mit_maske = ...
mit_query = ...
# print(len(mit_maske), len(mit_query))

In [ ]:
# Aufgabe 2: Stadtstaaten mit isin, danach die Raucher darunter zählen
# Tipp: df["bundesland"].isin([...]); raucher ist eine bool-Spalte, sum() zählt True
stadtstaaten = ...
anzahl_raucher = ...
# print(len(stadtstaaten), anzahl_raucher)

In [ ]:
# Aufgabe 3: BMI nachrechnen und mit der Spalte bmi vergleichen
# Tipp: groesse_m = df["groesse_cm"] / 100, am Ende .round(1)
df["bmi_neu"] = ...
# abweichung = (df["bmi_neu"] - df["bmi"]).abs()
# print(abweichung.max(), df["bmi_neu"].isna().sum())

In [ ]:
# Aufgabe 4: altersgruppe und bmi_klasse mit pd.cut, Normalgewichtige sortieren
# Tipp: bins sind die Grenzen, labels eine weniger als Grenzen, right=False
df["altersgruppe"] = ...
df["bmi_klasse"] = ...
# print(df["altersgruppe"].value_counts())
# print(df["bmi_klasse"].value_counts())
normal = ...
normal

## Block 5: Gruppieren

Die nächste Zelle legt `altersgruppe` noch einmal an. Wenn Sie Block 4 abgeschlossen haben, ändert sich nichts. Wenn nicht, können Sie hier trotzdem weitermachen.

1. Berechnen Sie die durchschnittliche Größe und das durchschnittliche Gewicht je Altersgruppe. **Erwartet: Erwachsen 173.3 cm und 77.4 kg, Senior 171.2 cm und 81.7 kg**
2. Berechnen Sie die mittleren Leistungsausgaben je Bundesland und Altersgruppe und stellen Sie sie mit `unstack()` als Kreuztabelle dar. **Erwartet: 16 Zeilen, 2 Spalten; Berlin: Erwachsen 2212, Senior 5998**
3. Bestimmen Sie den Raucheranteil je Geschlecht und je Bundesland (Tipp: Mittelwert einer bool-Spalte). **Erwartet: männlich 0.262, weiblich 0.224; höchster Anteil je Bundesland: Mecklenburg-Vorpommern mit 0.295**
4. Leistungsausgaben sind schief verteilt. Vergleichen Sie je Bundesland Mittelwert und Median. **Erwartet: insgesamt Mittelwert 3114.75 Euro, Median 1606.48 Euro; in jedem Bundesland liegt der Median deutlich unter dem Mittelwert**

In [ ]:
# Aufholzelle: altersgruppe sicher anlegen (identisch mit Block 4, Aufgabe 4)
df["altersgruppe"] = pd.cut(df["alter"], bins=[0, 18, 65, 120],
                            labels=["Kind", "Erwachsen", "Senior"], right=False)
df["altersgruppe"].value_counts()

In [ ]:
# Aufgabe 1: mittlere Größe und mittleres Gewicht je Altersgruppe
# Tipp: groupby("altersgruppe", observed=True), zwei Spalten in doppelter Klammer, mean, round(1)
groesse_gewicht = ...
groesse_gewicht

In [ ]:
# Aufgabe 2: mittlere Leistungsausgaben je Bundesland und Altersgruppe als Kreuztabelle
# Tipp: groupby mit einer Liste aus zwei Spalten, mean, round(0), unstack()
tabelle = ...
tabelle

In [ ]:
# Aufgabe 3: Raucheranteil je Geschlecht und je Bundesland
# Tipp: Der Mittelwert einer bool-Spalte ist der Anteil der True-Werte
raucher_geschlecht = ...
raucher_land = ...
# print(raucher_geschlecht)
# print(raucher_land.sort_values(ascending=False).head(3))

In [ ]:
# Aufgabe 4: Mittelwert gegen Median der Leistungsausgaben
# Tipp: agg(["count", "mean", "median"]) liefert mehrere Kennzahlen auf einmal
gesamt_mittel = ...
gesamt_median = ...
je_land = ...
je_land

## Block 6: Verbinden, Datum, Speichern

Die nächste Zelle legt eine kleine Nachschlagetabelle an: Bundesland und zugehörige Region. In der Praxis käme sie aus einer eigenen Datei.

1. Hängen Sie die Region mit `pd.merge` als Left Join an `df` an. Prüfen Sie die Zeilenzahl und zählen Sie die Versicherten je Region. **Erwartet: weiterhin 5025 Zeilen, keine fehlende Region; Ost 1746, West 1276, Nord 1128, Süd 875**
2. Bauen Sie eine zweite Tabelle mit 1000 zufällig gezogenen Versichertennummern und der Spalte `programm=True`. Hängen Sie sie per Left Join mit `indicator=True` an und zählen Sie die Treffer. **Erwartet: 1004 mal `both`, 4021 mal `left_only`** (Warum nicht genau 1000? Die Datei enthält einige doppelte Zeilen. Darum kümmern Sie sich im nächsten Lab.)
3. Wandeln Sie `geburtsdatum` in ein Datum um, leiten Sie `geburtsjahr` und `geburtsmonat` ab und zählen Sie die Versicherten je Geburtsmonat. **Erwartet: Geburtsjahre von 1926 bis 2008; die meisten Geburtstage im Dezember (481), die wenigsten im Februar (385)**
4. Speichern Sie eine Auswertung je Bundesland (Anzahl, Mittelwert und Median der Leistungsausgaben) als CSV mit Semikolon und Dezimalkomma in das temporäre Verzeichnis `TMP` und lesen Sie die Datei zur Kontrolle wieder ein. **Erwartet: 16 Zeilen, 4 Spalten**

In [ ]:
# Nachschlagetabelle: Bundesland -> Region
regionen = pd.DataFrame({
    "bundesland": [
        "Schleswig-Holstein", "Hamburg", "Bremen", "Niedersachsen", "Mecklenburg-Vorpommern",
        "Berlin", "Brandenburg", "Sachsen", "Sachsen-Anhalt", "Thüringen",
        "Nordrhein-Westfalen", "Hessen", "Rheinland-Pfalz", "Saarland",
        "Bayern", "Baden-Württemberg",
    ],
    "region": ["Nord"] * 5 + ["Ost"] * 5 + ["West"] * 4 + ["Süd"] * 2,
})

# temporäres Verzeichnis für alle Dateien, die dieses Lab schreibt
TMP = Path(tempfile.mkdtemp())
regionen.head()

In [ ]:
# Aufgabe 1: Region per Left Join anhängen
# Tipp: pd.merge(links, rechts, on="bundesland", how="left"); danach Zeilenzahl prüfen
mit_region = ...
# print(len(mit_region), mit_region["region"].isna().sum())
# mit_region["region"].value_counts()

In [ ]:
# Aufgabe 2: 1000 zufällige Versichertennummern, Left Join mit indicator=True
# Tipp: df["versicherten_nr"].drop_duplicates().sample(1000, random_state=1)
#       ergibt eine Series; .to_frame() macht daraus eine Tabelle, .assign(programm=True) ergänzt die Spalte
teilnahme = ...
gesamt = ...
# gesamt["_merge"].value_counts()

In [ ]:
# Aufgabe 3: Geburtsjahr und Geburtsmonat ableiten, je Monat zählen
# Tipp: pd.to_datetime, danach .dt.year und .dt.month; value_counts().sort_index()
df["geburtsdatum"] = ...
df["geburtsjahr"] = ...
df["geburtsmonat"] = ...
# print(df["geburtsjahr"].min(), df["geburtsjahr"].max())
# df["geburtsmonat"].value_counts().sort_index()

In [ ]:
# Aufgabe 4: Auswertung je Bundesland speichern und wieder einlesen
# Tipp: reset_index() macht aus dem Gruppierungsschlüssel eine normale Spalte.
#       to_csv(pfad, index=False, sep=";", decimal=",", encoding="utf-8-sig")
auswertung = ...
pfad = TMP / "auswertung_bundesland.csv"
...
# kontrolle = pd.read_csv(pfad, sep=";", decimal=",")
# print(pfad, kontrolle.shape)

## Zusatzaufgaben

1. Gruppen filtern: In welchen Bundesländern liegen die mittleren Leistungsausgaben über 3300 Euro? **Erwartet: 5 Bundesländer, an der Spitze Rheinland-Pfalz mit 3954 Euro**
2. Feinere Altersgruppen: Legen Sie mit `pd.cut` die Gruppen „18 bis 29", „30 bis 49", „50 bis 64", „65 bis 79" und „80 plus" an und bestimmen Sie je Gruppe Anzahl und Median der Leistungsausgaben. **Erwartet: Der Median steigt mit dem Alter, von 786 Euro (18 bis 29) auf 5550 Euro (80 plus)**
3. Kennzahl zurückspielen: Berechnen Sie den Median der Leistungsausgaben je Bundesland, hängen Sie ihn per `merge` als Spalte `median_land` an jede Zeile an und bestimmen Sie den Anteil der Versicherten, die über dem Median ihres Bundeslands liegen. **Erwartet: Anteil 0.499**

In [ ]:
# Zusatz 1: Bundesländer mit mittleren Leistungsausgaben über 3300 Euro
# Tipp: Das Ergebnis von groupby(...).mean() ist eine Series, die Sie mit einer Maske filtern
je_land_mittel = ...
ueber_3300 = ...
ueber_3300

In [ ]:
# Zusatz 2: feinere Altersgruppen, Anzahl und Median der Ausgaben
# Tipp: bins=[18, 30, 50, 65, 80, 120], right=False
df["altersgruppe_fein"] = ...
fein = ...
fein

In [ ]:
# Zusatz 3: Median je Bundesland an jede Zeile anhängen und vergleichen
# Tipp: groupby(...).median().rename("median_land").reset_index(), dann merge mit how="left"
median_land = ...
vergleich = ...
anteil_ueber = ...
anteil_ueber

## Was Sie mitnehmen

- NumPy und pandas rechnen mit ganzen Spalten. Masken mit `&`, `|`, `~` filtern Arrays und DataFrames auf dieselbe Weise, und der Mittelwert einer Maske ist ein Anteil.
- Das pandas-Grundrezept: einlesen, mit `shape`, `head`, `info` prüfen, filtern, Spalten ableiten, mit `groupby` und `agg` zusammenfassen. Bei schiefen Größen wie Ausgaben sagt der Median mehr als der Mittelwert.
- Nach jedem `merge` prüfen Sie die Zeilenzahl. Ergebnisse speichern Sie mit `to_csv`, für Excel mit Semikolon und Dezimalkomma.